# AICE 자료 — Colab T4 실측 벤치마크

이 노트북을 **Colab에서 `런타임 → T4 GPU`로 바꾼 뒤 위에서부터 실행**하세요.
각 도메인의 실제 소요 시간을 재서 마지막 셀에 표로 출력합니다.
그 표를 복사해 주시면 강의자료의 "T4 기준(추정)" 숫자를 실측으로 바꿉니다.

- 전체 약 15~25분 (BERT·이미지 각 3에폭 포함)
- 목적은 성능이 아니라 **시간**입니다.

In [ ]:
# 0. 환경 — BERT(TF)용 tf-keras 를 tensorflow import 전에
!pip install -q tf-keras "transformers>=4.40,<5" datasets

import os, time
import importlib.util
if importlib.util.find_spec('tf_keras'):
    os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus)
assert gpus, "T4 미연결! 런타임 → 런타임 유형 변경 → T4 GPU"

BENCH = {}
def rec(name, sec, note=''):
    BENCH[name] = {'sec': round(sec, 1), 'note': note}
    print(f'  [기록] {name}: {sec:.1f}s  {note}')

## 1. Tabular — 5모델 비교 + XGB GridSearch (Titanic 규모)

In [ ]:
import numpy as np, pandas as pd, seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
RS = 42
df = sns.load_dataset('titanic').drop(columns=['alive','deck','class','embark_town','who','adult_male','alone'])
y = df.pop('survived')
for c in df.columns:
    df[c] = df[c].fillna(df[c].median() if df[c].dtype.kind in 'biufc' else df[c].mode()[0])
    if df[c].dtype == 'object' or df[c].dtype.name == 'category':
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
X = StandardScaler().fit_transform(df)
Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=.2, random_state=RS, stratify=y)
models = {'LR': LogisticRegression(max_iter=1000), 'RF': RandomForestClassifier(n_estimators=200),
          'GB': GradientBoostingClassifier(n_estimators=200),
          'XGB': XGBClassifier(n_estimators=200, eval_metric='logloss'),
          'LGBM': LGBMClassifier(n_estimators=200, verbose=-1)}
t = time.time()
for m in models.values(): m.fit(Xtr, ytr)
rec('Tabular 5모델 학습', time.time() - t)
t = time.time()
GridSearchCV(XGBClassifier(eval_metric='logloss'),
             {'n_estimators': [100,200,300], 'max_depth': [4,6,8], 'learning_rate': [.01,.05,.1]},
             cv=5, n_jobs=-1).fit(Xtr, ytr)
rec('Tabular XGB GridSearch(27조합)', time.time() - t)

## 2. Text 방법 A — char_wb TF-IDF (NSMC 3만)

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as LR
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
tr = pd.read_csv('https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt', sep='\t').dropna().sample(30000, random_state=RS)
def clean(t): return re.sub(r'\s+', ' ', re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', str(t))).strip()
tr['document'] = tr['document'].map(clean)
tr = tr[tr['document'] != '']
t = time.time()
Xt = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=50000, min_df=2, sublinear_tf=True).fit_transform(tr['document'])
for m in [LR(max_iter=1000), LinearSVC(), MultinomialNB(alpha=.1)]: m.fit(Xt, tr['label'])
rec('Text A: char_wb TF-IDF + 3모델', time.time() - t, '(NSMC 3만)')

## 3. Text 방법 B — BERT klue/bert-base 파인튜닝 (6천건·3에폭)

In [ ]:
from transformers import BertTokenizerFast, TFBertForSequenceClassification
sub = tr.sample(6000, random_state=RS)
tok = BertTokenizerFast.from_pretrained('klue/bert-base')
enc = tok(sub['document'].tolist(), truncation=True, padding=True, max_length=128, return_tensors='tf')
ds = tf.data.Dataset.from_tensor_slices((dict(enc), sub['label'].tolist())).shuffle(1000, seed=RS).batch(32).prefetch(tf.data.AUTOTUNE)
t = time.time()
bert = TFBertForSequenceClassification.from_pretrained('klue/bert-base', num_labels=2, from_pt=True)
rec('BERT 로드(from_pt 변환)', time.time() - t)
bert.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
             loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t = time.time(); bert.fit(ds, epochs=3, verbose=1); e = time.time() - t
rec('BERT 3에폭(6천건·MAX128·B32)', e, f'에폭당 {e/3:.1f}s')

## 4. Image — MobileNetV2 2단계 전이학습 (flowers 5클래스·각 3에폭)

In [ ]:
import pathlib
p = pathlib.Path(tf.keras.utils.get_file('flower_photos',
    'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz', untar=True))
p = p if (p/'daisy').exists() else next(p.rglob('flower_photos/daisy')).parent
com = dict(label_mode='categorical', image_size=(224,224), batch_size=32, seed=RS)
trd = tf.keras.utils.image_dataset_from_directory(p, validation_split=.2, subset='training', **com).prefetch(tf.data.AUTOTUNE)
vad = tf.keras.utils.image_dataset_from_directory(p, validation_split=.2, subset='validation', **com).prefetch(tf.data.AUTOTUNE)
base = tf.keras.applications.MobileNetV2(input_shape=(224,224,3), weights='imagenet', include_top=False); base.trainable = False
inp = tf.keras.Input((224,224,3)); x = tf.keras.layers.Rescaling(1/127.5, offset=-1)(inp)
x = base(x, training=False); x = tf.keras.layers.GlobalAveragePooling2D()(x); x = tf.keras.layers.Dropout(.3)(x)
out = tf.keras.layers.Dense(5, activation='softmax')(x); mdl = tf.keras.Model(inp, out)
mdl.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
t = time.time(); mdl.fit(trd, validation_data=vad, epochs=3, verbose=1); e1 = time.time() - t
rec('Image Phase1 3에폭', e1, f'에폭당 {e1/3:.1f}s')
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
mdl.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
t = time.time(); mdl.fit(trd, validation_data=vad, epochs=3, verbose=1); e2 = time.time() - t
rec('Image Phase2 3에폭(파인튜닝)', e2, f'에폭당 {e2/3:.1f}s')

## 5. 결과 — 이 표를 복사해 주세요

In [ ]:
print('=' * 60)
print(f"{'항목':38} {'초':>8}  비고")
print('-' * 60)
for k, v in BENCH.items():
    print(f"{k:38} {v['sec']:>8.1f}  {v['note']}")
print('=' * 60)
be = BENCH.get('BERT 3에폭(6천건·MAX128·B32)', {})
print('\n[강의자료 갱신용 요약]')
print(f"  Text B(BERT) 6천건 3에폭 {be.get('sec','?')}s → 6만건 환산 에폭당 약 {be.get('sec',0)/3*10:.0f}s")
print(f"  Image 에폭당  P1 {BENCH.get('Image Phase1 3에폭',{}).get('sec',0)/3:.0f}s / P2 {BENCH.get('Image Phase2 3에폭(파인튜닝)',{}).get('sec',0)/3:.0f}s")